# Phase 5 — Construction des portefeuilles

## Pourquoi cette phase est importante

Les cours en CHF, l'univers convertible en CHF et le SARON sont déjà publiés. Cette phase en fait trois portefeuilles. Les trois utilisent les mêmes titres et les mêmes rendements mensuels. Seule la règle de poids change. On peut donc attribuer l'écart de résultat à la règle, et à elle seule.

### Portefeuille 1 — Équipondéré

Objectif : comprendre l'effet de la diversification simple.

Chaque titre retenu reçoit le même poids. Le rendement passé et le risque passé ne servent pas à choisir ce poids.

### Portefeuille 2 — Volatilité minimale

Objectif : comprendre les mécanismes de réduction du risque.

Les poids sont ceux qui rendent la variance du portefeuille la plus petite. Un titre calme, ou un titre qui ne baisse pas en même temps que les autres, reçoit davantage. Le calcul s'appuie sur la covariance des rendements mensuels en CHF.

### Portefeuille 3 — Sharpe maximal

Objectif : comprendre la notion de rendement ajusté au risque.

Les poids maximisent le rendement au-dessus du SARON, par unité de risque. Le portefeuille peut accepter une volatilité plus haute lorsque le rendement supplémentaire est plus que proportionnel.

## Les trois livrables

| Livrable | Fichier | Rôle |
|---|---|---|
| Documentation des stratégies | `documentation/05_portfolio_construction.ipynb` | La méthode, les formules, les fichiers produits |
| Calculs détaillés | `reports/portfolio/05_calculs_detailles.ipynb` | Le même calcul repris sur un mois, puis sur tous les mois de la fenêtre |
| Analyse comparative | `notebooks/05_analyse_comparative.ipynb` | La lecture des écarts entre les trois portefeuilles |

La vérification automatique est à part, dans `tests/portfolio/phase_05_validation.ipynb`. Elle ne remplace aucun des trois livrables.

## Entrées

Le module lit trois fichiers déjà traités. Il ne réécrit pas `data/01_raw`.

| Rôle | Fichier | Colonnes utilisées |
|---|---|---|
| Cours en CHF | `data/02_processed/market/market_prices_chf.csv` | `symbol`, `timestamp`, `close_chf` |
| Univers | `data/03_universe/eligible/chf_portfolio_universe.csv` | `yahoo_symbol`, `ticker`, `name`, `exchange`, `listing_currency`, `chf_status`, `exclusion_reason` |
| Taux sans risque | `data/02_processed/macro/saron.csv` | `series_id`, `date`, `value`, `rate_currency` |

`close_chf` est le cours déjà converti en francs, échelle de cotation comprise. Le SARON est le taux au jour le jour de la BNS, en pourcentage annuel, une observation par mois. Sa devise dans le fichier est le CHF.

Le statut ODD est vide sur tout le master. Il n'entre pas dans le choix des titres. Un titre retenu est un titre déjà convertible en CHF, avec un cours de fin de mois sur toute la fenêtre.

## Fenêtre commune

La longueur vient de `MARKET_HISTORY_YEARS` dans `src/config.py`. La valeur du projet est 10 ans. La fin est le dernier mois du SARON. Avec une fin $T$ et une longueur de $Y$ années, le premier mois de rendement est

$$
t_1 = T - 12Y + 1
$$

Pour calculer le rendement de $t_1$, il faut aussi le cours de fin du mois précédent. Les cours nécessaires vont donc de $t_1 - 1$ à $T$. Le nombre de mois de rendement est $12Y$. Le nombre de fins de mois de cours est $12Y + 1$. La cellule en bas de ce notebook affiche $T$, $t_1$ et ces deux comptes pour les fichiers présents. Un mois de SARON plus récent décale toute la fenêtre. Des cours plus récents que $T$ attendent le SARON de ce mois.

Un mois sans SARON dans cette fenêtre arrête le calcul. Le taux n'est pas interpolé.

## Qui entre dans les trois portefeuilles

Le cours de fin de mois du titre $i$ est le dernier `close_chf` observé pendant le mois $m$ :

$$
P_{i,m} = \text{dernier close CHF de } i \text{ dans le mois } m
$$

Un titre entre dans les trois portefeuilles lorsqu'il remplit toutes les conditions suivantes.

1. `chf_status` vaut `included` dans l'univers CHF. Sinon le motif reste celui de la phase 4, `no_snb_series`.
2. Il a au moins un cours dans l'historique. Sinon le motif est `no_price_history`.
3. Aucun cours de la fenêtre n'est négatif ou nul. Sinon le motif est `non_positive_close`.
4. Toutes les fins de mois exigées, de $t_1 - 1$ à $T$, sont présentes et strictement positives. Sinon le motif est `incomplete_monthly_history`.
5. La série de rendements n'est pas constante. Sinon le motif est `constant_monthly_return`.

Ces décisions sont écrites dans `portfolio_membership.csv`. Les trois stratégies reçoivent ensuite exactement la même liste, dans `decision = included`. Un titre écarté n'a aucun poids.

## Rendement mensuel d'un titre

Le rendement simple du mois $t$ utilise deux cours de fin de mois consécutifs :

$$
R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1
$$

$t$ et $t-1$ sont deux mois civils qui se suivent. Si l'un des deux cours manque, le rendement de ce mois n'est pas calculé et le titre sort de la liste. Aucun cours n'est remplacé par le précédent.

Le rendement est déjà en CHF, parce que $P$ est `close_chf`. Le change du mois est donc dans le rendement du mois où le taux SNB change.

## Taux sans risque

Le fichier SARON donne un pourcentage annuel. Le Sharpe mensuel a besoin d'un rendement sur le mois. Pour le mois $t$ :

$$
r_{f,t} = \left(1 + \frac{\mathrm{SARON}_t}{100}\right)^{1/12} - 1
$$

$\mathrm{SARON}_t$ est le taux du mois pendant lequel le rendement du portefeuille est gagné. Le rendement mesuré entre la fin du mois $t-1$ et la fin du mois $t$ retire le SARON du mois $t$.

Le facteur 252 du calendrier quotidien ne s'applique pas ici. La série commune est mensuelle, comme le SARON et comme le taux de change SNB. L'annualisation des métriques utilise donc 12.

L'excédent du portefeuille est

$$
E_{p,t} = R_{p,t} - r_{f,t}
$$

## Rendement du portefeuille

Les poids $w_i$ sont fixés pour toute la fenêtre. Chaque mois, le rendement du portefeuille est

$$
R_{p,t} = \sum_{i=1}^{N} w_i \, R_{i,t}
$$

C'est le rendement d'un portefeuille ramené à ces poids au début de chaque mois. Aucun coût de transaction n'est retiré.

Contraintes, pour les trois stratégies :

$$
w_i \ge 0, \qquad \sum_{i=1}^{N} w_i = 1
$$

Le portefeuille est donc entièrement investi, sans vente à découvert et sans levier. Après le solveur, un poids négatif dû seulement à l'arrondi est ramené à zéro, puis les poids sont redivisés par leur somme.

## Portefeuille 1 — Équipondéré

$$
w_i = \frac{1}{N}
$$

$N$ est le nombre de titres retenus. Le poids ne dépend pas de $R_{i,t}$.

Ce portefeuille sert de base. Il montre ce que produit le fait de tenir toute la liste au même poids. Le nombre effectif de titres, défini plus bas, est alors exactement $N$.

## Portefeuille 2 — Volatilité minimale

La covariance utilisée est la covariance d'échantillon des rendements mensuels, avec le diviseur $T - 1$ :

$$
\Sigma_{ij} = \frac{1}{T-1} \sum_{t=1}^{T} \left(R_{i,t} - \bar R_i\right)\left(R_{j,t} - \bar R_j\right)
$$

$T$ est le nombre de mois. Le programme cherche

$$
\min_w \; w^\top \Sigma w
$$

sous les contraintes $w_i \ge 0$ et $\sum_i w_i = 1$.

$w^\top \Sigma w$ est la variance mensuelle du rendement du portefeuille. Le solveur est SLSQP. Le point de départ est l'équipondéré, qui est un portefeuille autorisé : le minimum obtenu ne peut donc pas avoir une variance plus haute que lui.

La réduction du risque, dans cette phase, est cette baisse de variance. Elle passe par la covariance : un titre peut recevoir un poids faible même s'il est peu volatile, lorsqu'il bouge comme le reste du portefeuille. Elle passe aussi par la concentration : le minimum met l'essentiel du poids sur un sous-ensemble de titres. La volatilité annualisée publiée est l'écart-type de $R_{p,t}$, pas l'écart-type de l'excédent.

## Portefeuille 3 — Sharpe maximal

Le Sharpe mensuel d'un vecteur de poids est

$$
S(w) = \frac{w^\top \mu}{\sqrt{w^\top \Sigma_e w}}
$$

$\mu$ est la moyenne des rendements excédentaires $R_{i,t} - r_{f,t}$. $\Sigma_e$ est leur covariance d'échantillon, encore avec le diviseur $T - 1$.

$\Sigma_e$ n'est pas égale à $\Sigma$. Retirer un taux qui change au cours du temps modifie les variances. La volatilité minimale minimise la variance de $R_{p,t}$. Le Sharpe maximal utilise la variance de $E_{p,t}$.

$S(w)$ ne change pas si tous les poids sont multipliés par le même nombre positif. Le programme résout donc le problème équivalent

$$
\max_y \; \mu^\top y
\qquad
y^\top \Sigma_e y \le 1
\qquad
y \ge 0
$$

puis ramène la solution à une somme de 1 :

$$
w = \frac{y}{\sum_i y_i}
$$

SLSQP est lancé depuis le titre dont l'excédent moyen est le plus haut, puis depuis huit autres points de départ tirés avec la graine fixe 15. Le vecteur conservé est celui dont le Sharpe est le plus haut. Il doit au moins atteindre le Sharpe de l'équipondéré, puisque l'équipondéré est autorisé.

Ces poids sont estimés sur les mêmes mois que le Sharpe publié. Le chiffre mesure l'objectif sur cette fenêtre, une fois les rendements connus. Il ne mesure pas le résultat d'une règle qui aurait été figée au premier mois. À la prochaine exécution, un mois de plus change les poids de toute la fenêtre, y compris les mois déjà publiés. L'équipondéré, lui, ne lit pas les rendements pour fixer ses poids.

## Métriques publiées

Toutes les métriques sont calculées sur les $T$ mois de la fenêtre.

Rendement annualisé, géométrique :

$$
R_p^{\mathrm{ann}} = \left(\prod_{t=1}^{T} (1 + R_{p,t})\right)^{12/T} - 1
$$

Volatilité annualisée, écart-type d'échantillon des rendements du portefeuille :

$$
\sigma_p^{\mathrm{ann}} = \sqrt{ \frac{1}{T-1} \sum_{t=1}^{T} \left(R_{p,t} - \bar R_p\right)^2 } \times \sqrt{12}
$$

Sharpe annualisé. La moyenne et l'écart-type portent sur l'excédent $E_{p,t}$, pas sur $R_{p,t}$ :

$$
\mathrm{Sharpe}_p = \frac{\bar E_p}{s_E} \times \sqrt{12}
$$

$s_E$ est l'écart-type d'échantillon de $E_{p,t}$.

Baisse maximale. La richesse part de 1 avant le premier mois :

$$
V_t = \prod_{k=1}^{t} (1 + R_{p,k})
$$

$$
\mathrm{DD}_t = \frac{V_t}{\max_{k \le t} V_k} - 1
\qquad
\mathrm{MDD} = \min_t \mathrm{DD}_t
$$

Le nombre publié est négatif. Une valeur $-0{,}17$ décrit une perte de 17 % depuis un sommet.

Nombre effectif de titres :

$$
N_{\mathrm{eff}} = \frac{1}{\sum_{i=1}^{N} w_i^2}
$$

Pour l'équipondéré, $N_{\mathrm{eff}} = N$. Plus les poids se concentrent, plus $N_{\mathrm{eff}}$ baisse. Un seul titre donne $N_{\mathrm{eff}} = 1$.

## Création des CSV

Le code est `src/portfolio_construction/build_portfolios.py`. Depuis la racine du projet :

```text
python -m src.pipeline.refresh
```

Cette commande acquiert les jeux dont la fréquence du registre est due, remplace leur fichier brut, recharge la base, complète les devises de cotation manquantes, retraite les cours en CHF, reconstruit les trois portefeuilles et relance le contrôle qualité. Le détail de l'ordre et des fréquences est dans `documentation/03_data_acquisition.ipynb`.

Pour reconstruire seulement les cinq CSV, une fois les fichiers traités à jour :

```text
python -m src.portfolio_construction.build_portfolios
```

La fonction `build_portfolios` enchaîne les étapes suivantes.

1. Lire les cours CHF, l'univers et le SARON.
2. Garder le dernier cours de chaque titre et de chaque mois.
3. Fixer la fenêtre : $12 \times$ `MARKET_HISTORY_YEARS` mois, terminée au dernier mois du SARON.
4. Écrire la décision de chaque titre du master.
5. Calculer le panneau de rendements des titres retenus.
6. Calculer les trois vecteurs de poids.
7. Calculer, pour chaque vecteur, la série mensuelle et les métriques.
8. Écrire les cinq CSV.

`data/03_universe/` reste la liste des titres : candidat, éligible, exclu. Les poids, les rendements et les métriques sont une couche suivante. Ils sont écrits dans `data/04_portfolios/`. Chaque CSV est d'abord écrit dans un fichier temporaire du même dossier, puis ce fichier remplace la version précédente. Le notebook qui refait l'arithmétique est dans `reports/portfolio/`.

### `portfolio_membership.csv`

Une ligne par titre du master.

| Colonne | Sens |
|---|---|
| `yahoo_symbol`, `ticker`, `name`, `exchange`, `listing_currency` | Identité du titre |
| `decision` | `included` ou `excluded` |
| `reason` | Motif d'écart, vide lorsque le titre est retenu |
| `available_price_months` | Fins de mois strictement positives trouvées dans la fenêtre |
| `required_price_months` | Fins de mois exigées, soit $12Y + 1$ |

### `portfolio_weights.csv`

Une ligne par stratégie et par titre retenu. Un poids nul est conservé : il montre que le solveur a pu écarter un titre.

| Colonne | Sens |
|---|---|
| `strategy` | `equal_weight`, `minimum_volatility` ou `maximum_sharpe` |
| `yahoo_symbol`, `ticker`, `name`, `listing_currency` | Identité du titre |
| `weight` | Part du portefeuille, entre 0 et 1 |

### `portfolio_monthly_returns.csv`

Une ligne par mois et par stratégie. C'est le détail du calcul.

| Colonne | Sens |
|---|---|
| `month` | Mois du rendement, `YYYY-MM` |
| `strategy` | L'une des trois stratégies |
| `portfolio_return` | $R_{p,t}$ |
| `risk_free_return` | $r_{f,t}$, rendement mensuel du SARON |
| `excess_return` | $R_{p,t} - r_{f,t}$ |

### `portfolio_metrics.csv`

Une ligne par stratégie.

| Colonne | Sens |
|---|---|
| `strategy` | L'une des trois stratégies |
| `names` | Nombre de titres du vecteur de poids |
| `months` | Nombre de mois de rendement, soit $12Y$ |
| `window_start`, `window_end` | Premier et dernier mois de rendement |
| `annualized_return` | $R_p^{\mathrm{ann}}$ |
| `annualized_volatility` | $\sigma_p^{\mathrm{ann}}$, volatilité de $R_{p,t}$ |
| `sharpe_ratio` | Sharpe annualisé, excédent au-dessus du SARON |
| `maximum_drawdown` | Baisse maximale, nombre négatif |
| `maximum_weight` | Plus grand poids |
| `effective_holdings` | $N_{\mathrm{eff}}$ |
| `risk_free_series` | `SARON` |
| `risk_free_currency` | Devise de reporting du projet |

### `portfolio_comparison.csv`

Une ligne par métrique. Les deux dernières colonnes sont l'écart avec l'équipondéré : valeur de la stratégie moins valeur de l'équipondéré. Pour la volatilité, un écart négatif est une volatilité plus basse. Pour la baisse maximale, un écart positif rapproche le nombre de zéro, donc une perte depuis le sommet moins profonde.

| Colonne | Sens |
|---|---|
| `metric` | Nom de la métrique, le même que dans `portfolio_metrics.csv` |
| `equal_weight` | Valeur de l'équipondéré |
| `minimum_volatility` | Valeur de la volatilité minimale |
| `maximum_sharpe` | Valeur du Sharpe maximal |
| `minimum_volatility_minus_equal_weight` | Écart de la volatilité minimale |
| `maximum_sharpe_minus_equal_weight` | Écart du Sharpe maximal |

La cellule suivante lit ces fichiers et en montre la forme. Elle ne relance pas le calcul.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MARKET_HISTORY_YEARS, PORTFOLIO_DATA, REPORTING_CURRENCY

print(f"Devise de reporting : {REPORTING_CURRENCY}")
print(f"Années de fenêtre : {MARKET_HISTORY_YEARS}")
print()

for path in (
    PORTFOLIO_DATA / "portfolio_membership.csv",
    PORTFOLIO_DATA / "portfolio_weights.csv",
    PORTFOLIO_DATA / "portfolio_monthly_returns.csv",
    PORTFOLIO_DATA / "portfolio_metrics.csv",
    PORTFOLIO_DATA / "portfolio_comparison.csv",
):
    frame = pd.read_csv(path)
    print(f"{path.relative_to(PROJECT_ROOT)} : {len(frame)} lignes, colonnes = {', '.join(frame.columns)}")

membership = pd.read_csv(PORTFOLIO_DATA / "portfolio_membership.csv", dtype=str, keep_default_na=False)
print()
print(membership.groupby(["decision", "reason"], dropna=False).size().rename("titres").to_string())

metrics = pd.read_csv(PORTFOLIO_DATA / "portfolio_metrics.csv")
window = metrics.loc[:, ["strategy", "names", "months", "window_start", "window_end"]]
print()
print("Fenêtre de cette exécution")
print(window.to_string(index=False))
required = sorted(membership["required_price_months"].unique())
print(f"Fins de mois exigées : {', '.join(required)}")


Devise de reporting : CHF
Années de fenêtre : 10

data\04_portfolios\portfolio_membership.csv : 120 lignes, colonnes = yahoo_symbol, ticker, name, exchange, listing_currency, decision, reason, available_price_months, required_price_months
data\04_portfolios\portfolio_weights.csv : 249 lignes, colonnes = strategy, yahoo_symbol, ticker, name, listing_currency, weight
data\04_portfolios\portfolio_monthly_returns.csv : 360 lignes, colonnes = month, strategy, portfolio_return, risk_free_return, excess_return
data\04_portfolios\portfolio_metrics.csv : 3 lignes, colonnes = strategy, names, months, window_start, window_end, annualized_return, annualized_volatility, sharpe_ratio, maximum_drawdown, maximum_weight, effective_holdings, risk_free_series, risk_free_currency
data\04_portfolios\portfolio_comparison.csv : 6 lignes, colonnes = metric, equal_weight, minimum_volatility, maximum_sharpe, minimum_volatility_minus_equal_weight, maximum_sharpe_minus_equal_weight

decision  reason              

## Suite de la lecture

Le détail arithmétique, du cours de fin de mois jusqu'au Sharpe, est dans `reports/portfolio/05_calculs_detailles.ipynb`.

La lecture des écarts entre les trois règles de poids est dans `notebooks/05_analyse_comparative.ipynb`.

Le suivi et la performance liront `portfolio_weights.csv` et `portfolio_monthly_returns.csv`. Ces rendements mensuels sont recalculés avec les poids de la dernière estimation. Ils décrivent le mélange une fois les rendements connus. Ils ne sont pas l'historique d'un mandat figé au premier mois.